# Gate: Human-in-the-Loop with Custom Tools

Many workflows sit between "fully automate" and "always ask a human." Expense approval is a classic example: the agent handles clear cases on its own, but it should know when to escalate ambiguous ones for human review.

This notebook builds an expense approver around two custom tools:
- **`decide`** — for clear-cut cases (policy unambiguously allows or forbids)
- **`escalate`** — for ambiguous cases (near threshold, unclear category, suspicious notes)

Both tools round-trip through your application. `decide` logs the outcome; `escalate` puts the receipt in front of a human reviewer and waits for their decision before the agent continues.

| Part | Pattern | Use case |
|---|---|---|
| A | Agentic loop with inline tool handling | Development — everything in one process |
| B | Webhook-driven interrupt | Production — human review happens asynchronously |

## The interrupt mechanism

When the agent calls `escalate`, the loop **pauses** and hands control to your application:

```
agent calls escalate(receipt_id, question)
        ↓
loop intercepts — stop_reason: tool_use
        ↓
application shows question to human reviewer
        ↓
human decides → POST tool_result back
        ↓
agent resumes with human's decision in context
```

This is the same pattern as the standard messages API `tool_use` / `tool_result` cycle — the difference is that your code, not Claude, determines what happens between the call and the result.

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL = 'claude-sonnet-4-6';

console.log('Setup complete. Model:', MODEL);

## The decide vs escalate Pattern

Two tools divide every decision into a binary routing question:

```
Agent receives receipt
        │
        ▼
Is the policy outcome clear?
        │
   ┌────┴────┐
  Yes       No
   │         │
decide()  escalate()
approve/  → Human Review
reject    → Wait for decision
```

**Calibration matters.** An agent that escalates everything defeats automation; an agent that never escalates is dangerous. The system prompt encodes the boundary by listing the specific conditions that warrant escalation.

### Escalation trigger conditions

| Condition | Example |
|---|---|
| **Near threshold** | Amount $499; policy boundary is $500 |
| **Unclear category** | Receipt has no category, or category is ambiguous |
| **Suspicious notes** | Cash payment, no documentation, contradictory information |
| **Missing authorization** | High-value purchase without required approval on record |
| **Irreversible action** | Outcome cannot be undone if the decision turns out to be wrong |

These conditions belong in the system prompt — not the tool description — so the agent applies them as judgment, not mechanical rules.

In [ ]:
// Policy embedded as a constant — in production this would come from a file or database
const POLICY = `
Expense Approval Policy:
- Under $100: approve for meals, transport, or office supplies
- $100–$500: approve if category is clear (travel, equipment, software, training)
- Over $500: escalate for manager review
- Alcohol or entertainment: escalate if over $100
- Any missing or ambiguous category: escalate
- Cash payments or missing documentation: escalate
`.trim();

// Sample receipts — mix of clear and ambiguous cases
type Receipt = {
  id: string;
  amount: number;
  category: string;
  notes: string;
};

const RECEIPTS: Receipt[] = [
  { id: 'R-001', amount: 45,  category: 'meals',     notes: 'Team lunch' },
  { id: 'R-002', amount: 320, category: 'travel',    notes: 'Train ticket to client site' },
  { id: 'R-003', amount: 850, category: 'equipment', notes: 'External monitor' },
  { id: 'R-004', amount: 495, category: 'software',  notes: 'Annual SaaS license — renews next month' },
  { id: 'R-005', amount: 130, category: 'meals',     notes: 'Client dinner including wine' },
  { id: 'R-006', amount: 80,  category: '',          notes: 'No category provided' },
  { id: 'R-007', amount: 200, category: 'supplies',  notes: 'Paid vendor in cash, no receipt scan' },
];

// Tool definitions
const tools: Anthropic.Tool[] = [
  {
    name: 'decide',
    description: 'Record a final approve or reject for a clear-cut receipt.',
    input_schema: {
      type: 'object',
      properties: {
        receipt_id: { type: 'string', description: 'Receipt identifier' },
        action:     { type: 'string', enum: ['approve', 'reject'], description: 'Final decision' },
        reason:     { type: 'string', description: 'One-line reason for the decision' },
      },
      required: ['receipt_id', 'action', 'reason'],
    },
  },
  {
    name: 'escalate',
    description: 'Surface an ambiguous receipt for human review.',
    input_schema: {
      type: 'object',
      properties: {
        receipt_id: { type: 'string', description: 'Receipt identifier' },
        question:   { type: 'string', description: 'Specific question for the human reviewer' },
      },
      required: ['receipt_id', 'question'],
    },
  },
];

console.log(`Policy loaded. ${RECEIPTS.length} receipts queued for processing.`);

## Part A: Agentic Loop with Inline Tool Handling

The loop drives the agent until it has processed every receipt:

1. Send the policy + receipts as the first user message
2. Receive a response — the agent calls `decide` or `escalate` for each receipt
3. For each `tool_use` block:
   - `decide` → log the decision, return `{ recorded: true }`
   - `escalate` → run `simulateHumanReview()`, return `{ human_decision: '...' }`
4. Append the assistant response and all `tool_result` blocks to message history
5. Repeat until `stop_reason === 'end_turn'`

The `simulateHumanReview` function stands in for a real UI that would show the question to a human. In production (Part B), this function is replaced by an async webhook.

In [ ]:
// Simulate a human reviewer — in production this would be a real UI / async queue
function simulateHumanReview(receiptId: string, question: string): string {
  // Simple rule: reject if the agent flagged suspicious activity
  const isSuspicious = question.toLowerCase().includes('suspicious') ||
                       question.toLowerCase().includes('cash') ||
                       question.toLowerCase().includes('no receipt');
  const decision = isSuspicious ? 'reject' : 'approve';
  console.log(`  [human] ${receiptId}: ${decision}`);
  return decision;
}

const SYSTEM_PROMPT = `You are an expense approver. Review each receipt in the list against the provided policy.

For each receipt, make exactly ONE tool call:
- Call decide(receipt_id, action, reason) for clear-cut cases where the policy outcome is unambiguous
- Call escalate(receipt_id, question) for ambiguous cases:
  - Amount is near a policy threshold
  - Category is missing or unclear
  - Notes mention cash, missing documentation, or contradictory information
  - The item might qualify as alcohol or entertainment
  - The decision would be difficult or costly to reverse

Once you have called decide or escalate for a receipt, that receipt is finalized — do not revisit it.
Process all receipts exactly once, then stop.`;

const userMessage = `Policy:\n${POLICY}\n\nReceipts:\n${JSON.stringify(RECEIPTS, null, 2)}\n\nProcess all ${RECEIPTS.length} receipts.`;

type DecisionRecord = {
  receipt_id: string;
  lane: 'approve' | 'reject' | 'escalated';
  reason?: string;
  question?: string;
  human_decision?: string;
};

const decisions = new Map<string, DecisionRecord>();
let messages: Anthropic.MessageParam[] = [{ role: 'user', content: userMessage }];

console.log('=== Part A: running expense approver ===');
console.log('');

while (true) {
  const response = await client.messages.create({
    model: MODEL,
    max_tokens: 4096,
    system: SYSTEM_PROMPT,
    tools,
    messages,
  });

  const toolUseBlocks = response.content.filter(
    (b): b is Anthropic.ToolUseBlock => b.type === 'tool_use'
  );

  if (toolUseBlocks.length === 0) break;

  const toolResults: Anthropic.ToolResultBlockParam[] = [];

  for (const block of toolUseBlocks) {
    const args = block.input as Record<string, string>;
    const receiptId = args['receipt_id'];
    let result: Record<string, unknown>;

    if (block.name === 'decide') {
      const action = args['action'] as 'approve' | 'reject';
      decisions.set(receiptId, { receipt_id: receiptId, lane: action, reason: args['reason'] });
      console.log(`decide   → ${receiptId}: ${action} — "${args['reason']}"`);
      result = { recorded: true };

    } else if (block.name === 'escalate') {
      console.log(`escalate → ${receiptId}: "${args['question']}"`);
      const humanDecision = simulateHumanReview(receiptId, args['question']);
      decisions.set(receiptId, {
        receipt_id: receiptId,
        lane: 'escalated',
        question: args['question'],
        human_decision: humanDecision,
      });
      result = { human_decision: humanDecision };

    } else {
      result = { error: `unknown tool: ${block.name}` };
    }

    toolResults.push({
      type: 'tool_result',
      tool_use_id: block.id,
      content: JSON.stringify(result),
    });
  }

  // Append assistant response + tool results before the next loop iteration
  messages = [
    ...messages,
    { role: 'assistant', content: response.content } as Anthropic.MessageParam,
    { role: 'user', content: toolResults } as Anthropic.MessageParam,
  ];

  if (response.stop_reason === 'end_turn') break;
}

// Summary
console.log('');
console.log(`=== ${decisions.size}/${RECEIPTS.length} decisions ===`);
const laneCount: Record<string, number> = {};
for (const d of decisions.values()) {
  laneCount[d.lane] = (laneCount[d.lane] ?? 0) + 1;
}
console.log(Object.entries(laneCount).map(([k, v]) => `${k}: ${v}`).join('   '));

## Examining the Decisions

The table below shows which lane each receipt took and why. This makes the agent's calibration visible — you can verify that near-threshold cases were escalated and clear cases were decided autonomously.

In [ ]:
// Print a decision table for review
console.log('Receipt  | Lane      | Outcome  | Detail');
console.log('---------|-----------|----------|-------');

for (const receipt of RECEIPTS) {
  const d = decisions.get(receipt.id);
  if (!d) {
    console.log(`${receipt.id}   | MISSING   |          | not processed`);
    continue;
  }

  const outcome = d.lane === 'escalated' ? d.human_decision ?? '?' : d.lane;
  const detail  = d.lane === 'escalated'
    ? `escalated: "${d.question}"`
    : `decided:   "${d.reason}"`;

  console.log(`${receipt.id}   | ${d.lane.padEnd(9)} | ${outcome.padEnd(8)} | ${detail}`);
}

## Part B: Production — Webhook-Driven Interrupt

Part A holds the HTTP connection open while a human thinks — this works during development but does not scale. The production pattern replaces the inline `simulateHumanReview()` call with an asynchronous round-trip:

```
1. Agent calls escalate()
        ↓
2. Your server receives the tool_use event
        ↓
3. Your server writes the pending escalation to a queue / database
        ↓
4. Human reviewer opens the review UI, sees the question, decides
        ↓
5. UI POSTs { tool_result } back to your server
        ↓
6. Your server sends the tool_result to the messages API with the conversation history
        ↓
7. Agent resumes — sees human_decision in context, continues with remaining receipts
```

The code that responds to the agent (step 6) is identical to Part A — only the trigger changes (UI callback instead of inline function call). Conversation state (`messages` array) must be persisted between steps 3 and 6.

### What to persist

| State | Where to store |
|---|---|
| `messages` array (full conversation history) | Database / cache |
| Pending `tool_use_id` values | Alongside the escalation record |
| Session identifier | Returned to the reviewer so they can POST back |

This is the same `tool_use` → `tool_result` mechanism as standard tool use — the only difference is that the time between them may be minutes or hours instead of milliseconds.

## Summary

### The decide vs escalate pattern

| Tool | When to call | Agent's contract |
|---|---|---|
| `decide(id, action, reason)` | Policy outcome is unambiguous | One call per receipt, final |
| `escalate(id, question)` | Any of the five trigger conditions | Pause and wait for human |

### Five escalation trigger conditions

| Condition | Why it warrants escalation |
|---|---|
| Near threshold | Small errors in reading the amount could flip the decision |
| Unclear category | Policy rules are category-specific; wrong category = wrong rule |
| Suspicious notes | Agent cannot verify intent or authenticity |
| Missing authorization | High-stakes action requires a human signature |
| Irreversible action | Cannot undo if the decision turns out to be wrong |

### The interrupt mechanism

```
stop_reason: 'tool_use'  →  one or more tool_use blocks in content
  ├─ block.name === 'decide'   →  log + return { recorded: true }
  └─ block.name === 'escalate' →  human review → return { human_decision: '...' }
Append assistant + tool_result messages → next iteration
stop_reason: 'end_turn'  →  all receipts processed, exit loop
```

### Development vs production

| | Part A (development) | Part B (production) |
|---|---|---|
| Human review | Inline function call | Async UI + webhook callback |
| Connection | Single streaming session | Stateless — messages persisted between calls |
| Latency | Milliseconds | Minutes to hours |
| Code difference | `simulateHumanReview()` inline | Same `tool_result` POST, triggered by UI callback |